In [5]:

import sys
from pathlib import Path
root_path = "/Users/brianlerner/Library/CloudStorage/OneDrive-Personal/Code/Cerebra/src"
# root_path = str(Path(__file__).resolve().parent.parent)
sys.path.append(root_path)

from llm.chat import json_request
from openai import OpenAI

In [6]:
# def variations(keystring, description=None, n=5):
#     input_string = f"""Here is the input: {keystring}"""
#     if description:
#         input_string += f"\nHere is the description: {description}"

#     prompt = f"""I am going to give you a word/phrase. I would like you to generate {n} variations of the word/phrase that are semantically similar to the input but are phrased differently.
# Any words/phrases in parentheses in the input provide useful context for the meaning of the rest of that particular string.
# The generated items should be phrased in a natural way, i.e. they should not have parentheses or formal punctuation even if members of the input list do.
# The description, if present, is there to provide further context to aid in generating the variations.
# Please return the new words/phrases as a JSON formatted as {{'variations': ['item1', 'item2', 'item3', ...]}}.
# {input_string}
# Go!
# """

system = """You are part of an application that allows a user to explore their data by querying with natural language. After user input, different data elements are extracted and used to find corresponding elements present in the user's database using a vector search. However, the user's phrasing may not match up exactly with the stored name for a given element, causing the search to fail. Your specific task is to take the information stored for an element, and generate a word or phrase that reflects the way a user would refer to it. 

Return a JSON with key "output" and the generated string as the value. The string can include commonly accepted synonyms. Based on the context provided by the element information, you may use verbs or adjectives to better reflect the style of the user's query. If present, the key 'parent_name' in the element information indicates that the element is a sub-element of another element.

Here is an example:
Example element information:
{
    "element_type": "event",
    "name": "basketball game",
    "category" : "activity",
    "description" :  "Playing in game for the Duke basketball team."
}
Example outputs:
"playing in the basketball game"
"the basketball game"
"my bball game"
"""

def variations(element):
    import json

    event = json.dumps(element, indent=4)
    element_prompt = f"""
Your task is beginning now. Make sure to return a JSON with one key, "output" with the generated string as the value. Here is the element information:
{event}

Go!
"""

    prompt = system + element_prompt
    return prompt


In [7]:
temperatures = [0, 1, 2]
models = ["gpt-3.5-turbo-1106", "gpt-4-1106-preview", "gpt-4-0125-preview"]

In [8]:
import itertools
import pandas as pd

event = {
    "name": "headache",
    "category": "sensation",
    "description": "Dull pain on the left side of the head.",
}

n = 5
prompt = variations(event)
trials = []
for temperature, model in itertools.product(temperatures, models):
    for i in range(n): 
        print(f"Trial {i} with temperature {temperature} and model {model}")
        trials.append(
            {
                "trial": i,
                "temperature": temperature,
                "model": model,
                "prompt": prompt,
                "output": json_request(prompt, temperature=temperature, model=model),
            }
        )



Trial 0 with temperature 0 and model gpt-3.5-turbo-1106
Trial 1 with temperature 0 and model gpt-3.5-turbo-1106
Trial 2 with temperature 0 and model gpt-3.5-turbo-1106
Trial 3 with temperature 0 and model gpt-3.5-turbo-1106
Trial 4 with temperature 0 and model gpt-3.5-turbo-1106
Trial 0 with temperature 0 and model gpt-4-1106-preview
Trial 1 with temperature 0 and model gpt-4-1106-preview
Trial 2 with temperature 0 and model gpt-4-1106-preview
Trial 3 with temperature 0 and model gpt-4-1106-preview
Trial 4 with temperature 0 and model gpt-4-1106-preview
Trial 0 with temperature 0 and model gpt-4-0125-preview
Trial 1 with temperature 0 and model gpt-4-0125-preview
Trial 2 with temperature 0 and model gpt-4-0125-preview
Trial 3 with temperature 0 and model gpt-4-0125-preview
Trial 4 with temperature 0 and model gpt-4-0125-preview
Trial 0 with temperature 1 and model gpt-3.5-turbo-1106
Trial 1 with temperature 1 and model gpt-3.5-turbo-1106
Trial 2 with temperature 1 and model gpt-3.5-tur

In [11]:
df = pd.DataFrame(trials)
df.set_index(["trial", "temperature", "model"])

,,,prompt,output
trial,temperature,model,,
0,0,gpt-3.5-turbo-1106,You are part of an application that allows a u...,"{\n ""output"": ""dull pain in my head""\n}"
1,0,gpt-3.5-turbo-1106,You are part of an application that allows a u...,"{\n ""output"": ""dull pain in my head""\n}"
2,0,gpt-3.5-turbo-1106,You are part of an application that allows a u...,"{\n ""output"": ""dull pain in my head""\n}"
3,0,gpt-3.5-turbo-1106,You are part of an application that allows a u...,"{\n ""output"": ""dull pain in my head""\n}"
4,0,gpt-3.5-turbo-1106,You are part of an application that allows a u...,"{\n ""output"": ""dull pain in my head""\n}"
0,0,gpt-4-1106-preview,You are part of an application that allows a u...,"{\n ""output"": ""my headache on the left side..."
1,0,gpt-4-1106-preview,You are part of an application that allows a u...,"{\n ""output"": ""my headache on the left side..."
2,0,gpt-4-1106-preview,You are part of an application that allows a u...,"{\n ""output"": ""my headache on the left side..."
3,0,gpt-4-1106-preview,You are part of an application that allows a u...,"{\n ""output"": ""my headache on the left side..."


In [65]:
# GPT_MODEL = "gpt-3.5-turbo-1106"
# GPT_MODEL = "gpt-4-1106-preview"
GPT_MODEL = "gpt-4-0125-preview"
prompt = variations(event)
json_request(prompt, model=GPT_MODEL, temperature=2)

{'output': 'headache on the left side'}

In [ ]:
def generate_variations(keystring, description=None, n=5):
    # prompt = pr.generate.variations(keystring, description=description, n=n)
    prompt = variations(keystring, description=description, n=n)

    variations = json_request(prompt)

    if "variations" not in variations:
        raise ValueError("No variations found")
    if not isinstance(variations["variations"], list):
        raise ValueError("Variations is not a list")
    if len(variations["variations"]) != n:
        raise ValueError("Incorrect number of variations generated")

    return variations["variations"]